In [1]:
%pip install pandas numpy openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


ETAPA 0 — Conversão Excel para CSV

In [2]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

arquivo_origem = "Relatório de Fechamento.xlsx"

df = pd.read_excel(arquivo_origem)

ETAPA 1 — Filtro Fluxo de Caixa

In [3]:
df.columns = df.columns.str.strip()

df = df[df["Objeto"].str.contains("fluxo de caixa", case=False, na=False)]

df.to_excel("relatorio.xlsx", index=False)

print(f"Total de linhas após filtro: {len(df)}")

Total de linhas após filtro: 43643


ETAPA 2 — Tratamento (AY / AZ)

In [4]:
df = pd.read_excel("relatorio.xlsx")

total_inicial = len(df)

col_AY = "Valor Pedido Objeto Corrigido"
col_AZ = "Valor Pedido"

mask_remove = (
    ((df[col_AY].round(2) == 0.01) & (df[col_AZ].round(2) == 0.01)) |
    ((df[col_AY].round(2) == 0.22) & (df[col_AZ].round(2) == 0.22))
)

mask_alerta = (
    (df[col_AY].round(2).isin([0.01, 0.22])) ^
    (df[col_AZ].round(2).isin([0.01, 0.22]))
)

alertas = df[mask_alerta].copy()
alertas.to_excel("alertas_valor.xlsx", index=False) #avisa se um for significativo e outro não

df = df[~(mask_remove | mask_alerta)]

total_final = len(df)

df.to_excel("relatorio.xlsx", index=False)

print("ETAPA 3 concluída ✔")
print(f"Total inicial de linhas: {total_inicial}")
print(f"Linhas enviadas para alertas: {len(alertas)}")
print(f"Total final após tratamento: {total_final}")
print(f"Total removido da base principal: {total_inicial - total_final}")

ETAPA 3 concluída ✔
Total inicial de linhas: 43643
Linhas enviadas para alertas: 2625
Total final após tratamento: 41003
Total removido da base principal: 2640


ETAPA 3 - Excluir por Natureza

In [5]:
import pandas as pd

# Ler relatório original
df = pd.read_excel("relatorio.xlsx")

# =========================
# CONDIÇÃO DE EXCLUSÃO
# =========================

cond_excluir = (
    (
        df["Natureza"].isin([
            "Administrativo - Gerencial",
            "Administrativo - Protestos",
            "Criminal"
        ])
        |
        (df["Natureza Financeira"].astype(str).str.upper() == "NEUTRO")
        |
        (df["Tipo de Ação"].astype(str).str.upper() == "NOTIFICAÇÃO EXTRAJUDICIAL")
        |
        (df["Assunto Principal"].astype(str).str.upper() == "RESCISÃO CONTRATUAL - LOCAÇÃO")
    )
    &
    (df["Valor Pedido Objeto Corrigido"] < 1)
)

# =========================
# APLICAR FILTRO
# =========================

relatorio_limpo = df[~cond_excluir]

# =========================
# SALVAR NOVO RELATÓRIO
# =========================

relatorio_limpo.to_excel("RELATORIO_FILTRADO.xlsx", index=False)

print("Relatório filtrado criado com sucesso.")
print("Registros originais:", len(df))
print("Registros após filtro:", len(relatorio_limpo))
print("Registros removidos:", len(df) - len(relatorio_limpo))

Relatório filtrado criado com sucesso.
Registros originais: 41003
Registros após filtro: 38994
Registros removidos: 2009


ETAPA 4 - Obter o Entradas

In [6]:
import pandas as pd

# Datas do filtro
data_inicio = "2026-03-26"  # ALTERE AQUI
data_fim = "2026-04-25"   # ALTERE AQUI

# Ler planilha original
df = pd.read_excel("RELATORIO_FILTRADO.xlsx")

# Converter coluna de data
df["Data de cadastro"] = pd.to_datetime(df["Data de cadastro"], errors="coerce")


# =========================
# REMOVER DUPLICADAS
# =========================

df_sem_dup = df.drop_duplicates(subset="Pasta")

# =========================
# FILTRAR ENTRE DATAS
# =========================

filtro = (
    (df_sem_dup["Data de cadastro"] >= data_inicio) &
    (df_sem_dup["Data de cadastro"] <= data_fim)
)

entradas = df_sem_dup[filtro]

# =========================
# SALVAR PLANILHA
# =========================

entradas.to_excel("ENTRADAS.xlsx", index=False)

print("Planilha 'ENTRADAS.xlsx' criada com sucesso.")
print("Total de registros:", len(entradas))

Planilha 'ENTRADAS.xlsx' criada com sucesso.
Total de registros: 13


ETAPA 5 - Entradas Pos BP 

In [ ]:
import pandas as pd

# Data do filtro
data_bp = "2025-12-26"   # ALTERE AQUI
data_limite = "2026-04-25"   # ALTERE AQUI
 
# Ler planilha original
df = pd.read_excel("RELATORIO_FILTRADO.xlsx")

# Converter coluna de data
df["Data de cadastro"] = pd.to_datetime(df["Data de cadastro"], errors="coerce")


# =========================
# REMOVER DUPLICADAS
# =========================

df_sem_dup = (
    df.sort_values("Data de cadastro", ascending=False)
      .drop_duplicates(subset="Pasta")
)
# =========================
# FILTRAR APÓS BP
# =========================

filtro = (
    (df_sem_dup["Data de cadastro"] >= data_bp) &
    (df_sem_dup["Data de cadastro"] <= data_limite)
)
pos_bp = df_sem_dup[filtro]

# =========================
# SALVAR PLANILHA
# =========================

pos_bp.to_excel("POS_BP.xlsx", index=False)

print("Planilha 'POS_BP.xlsx' criada com sucesso.")
print("Total de registros:", len(pos_bp))

Planilha 'POS_BP.xlsx' criada com sucesso.
Total de registros: 111


ETAPA 6 — Corte de Data

In [ ]:
df = pd.read_excel("RELATORIO_FILTRADO.xlsx")

total_inicial = len(df)

col_data = "Data de cadastro"

def conv_data(col):

    # tenta converter datas normais
    d = pd.to_datetime(col, dayfirst=True, errors="coerce")

    # tenta converter datas Excel
    col_numeric = pd.to_numeric(col, errors="coerce")

    mask_excel = d.isna() & col_numeric.notna()

    d.loc[mask_excel] = pd.to_datetime(
        col_numeric[mask_excel],
        unit="D",
        origin="1899-12-30"
    )

    return d

df[col_data] = conv_data(df[col_data])

# ============================================
# >>> ALTERAR AQUI <<<
data_corte_manual = "25/04/2026"
# ============================================

data_corte = pd.to_datetime(data_corte_manual, dayfirst=True)

mask = df[col_data] >= data_corte

linhas_removidas = mask.sum()

df = df[~mask].copy()

total_final = len(df)

df.to_excel("RELATORIO_FILTRADO.xlsx", index=False)

print("ETAPA 5 concluída ✔")
print(f"Data de corte aplicada: {data_corte.date()}")
print(f"Total inicial de linhas: {total_inicial}")
print(f"Linhas removidas (>= data de corte): {linhas_removidas}")
print(f"Total final do documento: {total_final}")

ETAPA 5 concluída ✔
Data de corte aplicada: 2026-04-25
Total inicial de linhas: 38994
Linhas removidas (>= data de corte): 1
Total final do documento: 38993


ETAPA 7 — Separar por Status

In [9]:
df = pd.read_excel("RELATORIO_FILTRADO.xlsx")

total_inicial = len(df)

col_status = "Status"

# normalizar texto
df[col_status] = df[col_status].astype(str).str.strip().str.upper()

df_ativos = df[df[col_status] == "ATIVOS"]
df_baixa = df[df[col_status] == "BAIXA PROVISÓRIA"]
df_encerrados = df[df[col_status] == "ENCERRADOS"]

df_outros = df[~df[col_status].isin([
    "ATIVOS",
    "BAIXA PROVISÓRIA",
    "ENCERRADOS"
])]

df_ativos.to_excel("ATIVOS.xlsx", index=False)
df_baixa.to_excel("BAIXA_PROVISORIA.xlsx", index=False)
df_encerrados.to_excel("ENCERRADOS.xlsx", index=False)

print("ETAPA 6 concluída ✔")
print(f"Total de linhas na base: {total_inicial}")
print(f"ATIVOS: {len(df_ativos)}")
print(f"BAIXA PROVISÓRIA: {len(df_baixa)}")
print(f"ENCERRADOS: {len(df_encerrados)}")
print(f"OUTROS STATUS: {len(df_outros)}")

ETAPA 6 concluída ✔
Total de linhas na base: 38993
ATIVOS: 36859
BAIXA PROVISÓRIA: 975
ENCERRADOS: 1159
OUTROS STATUS: 0


ETAPA 8 - Gestão das duplicadas

In [10]:
import pandas as pd

# Carregar planilhas
ativos = pd.read_excel("ATIVOS.xlsx")
baixa_prov = pd.read_excel("BAIXA_PROVISORIA.xlsx")
encerrados = pd.read_excel("ENCERRADOS.xlsx")

# Garantir que a data está no formato correto
ativos["Data Cálculo"] = pd.to_datetime(ativos["Data Cálculo"])
baixa_prov["Data Cálculo"] = pd.to_datetime(baixa_prov["Data Cálculo"])
encerrados["Data Cálculo"] = pd.to_datetime(encerrados["Data Cálculo"])

# =========================
# ATIVOS → manter 2 mais recentes com datas diferentes
# =========================
ativos_tratado = (
    ativos.sort_values("Data Cálculo", ascending=False)
          .drop_duplicates(subset=["Pasta", "Data Cálculo"])
          .groupby("Pasta")
          .head(2)
)

# =========================
# BAIXA PROVISÓRIA → manter 1 mais recente
# =========================
baixa_prov_tratado = (
    baixa_prov.sort_values("Data Cálculo", ascending=False)
              .groupby("Pasta")
              .head(1)
)

# =========================
# ENCERRADOS → manter 1 mais recente
# =========================
encerrados_tratado = (
    encerrados.sort_values("Data Cálculo", ascending=False)
              .groupby("Pasta")
              .head(1)
)

# =========================
# SALVAR SOBRESCREVENDO OS ARQUIVOS
# =========================
ativos_tratado.to_excel("ATIVOS.xlsx", index=False)
baixa_prov_tratado.to_excel("BAIXA_PROVISORIA.xlsx", index=False)
encerrados_tratado.to_excel("ENCERRADOS.xlsx", index=False)

print(f"ATIVOS: {len(ativos_tratado)}")
print(f"BAIXA PROVISÓRIA: {len(baixa_prov_tratado)}")
print(f"ENCERRADOS: {len(encerrados_tratado)}")
print("Arquivos atualizados com sucesso ✔")

ATIVOS: 9335
BAIXA PROVISÓRIA: 256
ENCERRADOS: 375
Arquivos atualizados com sucesso ✔


ETAPA 9 - Detecção de Variação nos Ativos

In [11]:
import pandas as pd
import numpy as np

df = pd.read_excel("ATIVOS.xlsx")

df["Data Cálculo"] = pd.to_datetime(df["Data Cálculo"])

# ordenar por processo e data
df = df.sort_values(["Pasta", "Data Cálculo"], ascending=[True, False])

# pegar valor do mês anterior
df["valor_anterior"] = df.groupby("Pasta")["Valor Pedido Objeto Corrigido"].shift(-1)

# manter apenas registros que possuem comparação
comparacao = df[df["valor_anterior"].notna()].copy()

# calcular variação percentual
comparacao["variacao"] = (
    (comparacao["Valor Pedido Objeto Corrigido"] - comparacao["valor_anterior"]) /
    comparacao["valor_anterior"]
)

# identificar casos onde ambos valores são menores que 1
ambos_menor_1 = (
    (comparacao["Valor Pedido Objeto Corrigido"] < 1) &
    (comparacao["valor_anterior"] < 1)
)

# separar classificações
sem_variacao = comparacao[
    (comparacao["variacao"] == 0) | ambos_menor_1
]

com_variacao = comparacao[
    (~ambos_menor_1) &
    (comparacao["variacao"].abs() > 0) &
    (comparacao["variacao"].abs() < 0.05)
]

mudanca_prognostico = comparacao[
    (~ambos_menor_1) &
    (comparacao["variacao"].abs() >= 0.05)
]


# garantir ordenação
sem_variacao = (
    sem_variacao.sort_values("Data Cálculo", ascending=False)
                .groupby("Pasta")
                .head(1)
)

com_variacao = (
    com_variacao.sort_values("Data Cálculo", ascending=False)
                .groupby("Pasta")
                .head(1)
)

mudanca_prognostico = (
    mudanca_prognostico.sort_values("Data Cálculo", ascending=False)
                       .groupby("Pasta")
                       .head(1)
)

# salvar planilhas
sem_variacao.to_excel("ATIVOS_SEM_VARIACAO.xlsx", index=False)
com_variacao.to_excel("ATIVOS_COM_VARIACAO.xlsx", index=False)
mudanca_prognostico.to_excel("MUDANCA_DE_PROGNOSTICO.xlsx", index=False)
print(f"ATIVOS_SEM_VARIACAO: {len(sem_variacao)}")
print(f"ATIVOS_COM_VARIACAO: {len(com_variacao)}")
print(f"MUDANCA_DE_PROGNOSTICO: {len(mudanca_prognostico)}")

ATIVOS_SEM_VARIACAO: 1079
ATIVOS_COM_VARIACAO: 3525
MUDANCA_DE_PROGNOSTICO: 57


ETAPA 10 - Criação do Settled

In [12]:
import pandas as pd

# ===============================
# ETAPA 7 — Criação do SETTLED
# ===============================

print("Carregando planilhas...")

baixa = pd.read_excel("BAIXA_PROVISORIA.xlsx")
enc = pd.read_excel("ENCERRADOS.xlsx")

# remove espaços em nomes de colunas (muito comum no Excel)
baixa.columns = baixa.columns.str.strip()
enc.columns = enc.columns.str.strip()

# colunas utilizadas
col_data_baixa = "Data"
col_data_enc = "Data de Encerramento"

col_motivo_bp = "Motivo da Baixa Provisória"
col_motivo_enc = "Motivo Encerramento"


# ===============================
# FUNÇÃO ROBUSTA DE CONVERSÃO DE DATA
# ===============================

def conv_data(col):

    # tentativa padrão
    d = pd.to_datetime(col, dayfirst=True, errors="coerce")

    col_numeric = pd.to_numeric(col, errors="coerce")

    # trata números Excel (dias desde 1899)
    mask_excel = (
        d.isna() &
        col_numeric.notna() &
        (col_numeric > 0) &
        (col_numeric < 60000)
    )

    d.loc[mask_excel] = pd.to_datetime(
        col_numeric[mask_excel],
        unit="D",
        origin="1899-12-30"
    )

    # trata timestamp em nanossegundos
    mask_ns = (
        d.isna() &
        col_numeric.notna() &
        (col_numeric > 1e12)
    )

    d.loc[mask_ns] = pd.to_datetime(
        col_numeric[mask_ns],
        unit="ns"
    )

    return d


# ===============================
# CONVERSÃO DAS DATAS CORRETAS
# ===============================

print("Convertendo datas...")

# Função auxiliar para garantir que convertemos a coluna apenas se ela existir no df
def converter_se_existir(df, col):
    if col in df.columns:
        df[col] = conv_data(df[col])
        df[col] = df[col].dt.normalize() # já remove as horas

# Aplicando para a planilha BAIXA
converter_se_existir(baixa, col_data_baixa)
converter_se_existir(baixa, col_data_enc)

# Aplicando para a planilha ENCERRADOS
converter_se_existir(enc, col_data_baixa)
converter_se_existir(enc, col_data_enc)


# ===============================
# DIAGNÓSTICO DAS DATAS
# ===============================

print("\nDiagnóstico de datas")

if col_data_baixa in baixa.columns:
    print("BAIXA (Data) válidas:", baixa[col_data_baixa].notna().sum())
if col_data_enc in baixa.columns:
    print("BAIXA (Encerramento) válidas:", baixa[col_data_enc].notna().sum())

if col_data_baixa in enc.columns:
    print("ENCERRADOS (Data) válidas:", enc[col_data_baixa].notna().sum())
if col_data_enc in enc.columns:
    print("ENCERRADOS (Encerramento) válidas:", enc[col_data_enc].notna().sum())


# ===============================
# PERÍODO MANUAL
# ===============================

data_inicio_manual = "26/03/2026"  # ALTERE AQUI
data_fim_manual = "25/04/2026"  # ALTERE AQUI

ini = pd.to_datetime(data_inicio_manual, dayfirst=True)
fim = pd.to_datetime(data_fim_manual, dayfirst=True)


# ===============================
# FILTROS CORRETOS (Com verificação e flag de exceção)
# ===============================

print("\nAplicando filtro de período...")

def filtrar_duas_datas(df, col1, col2, data_ini, data_fim):
    d1 = df[col1] if col1 in df.columns else pd.Series([pd.NaT]*len(df), index=df.index)
    d2 = df[col2] if col2 in df.columns else pd.Series([pd.NaT]*len(df), index=df.index)
    
    # Validações individuais (se a data está no período)
    d1_in = (d1 >= data_ini) & (d1 <= data_fim)
    d2_in = (d2 >= data_ini) & (d2 <= data_fim)
    
    # Verificação de quais estão preenchidas
    d1_preenchida = d1.notna()
    d2_preenchida = d2.notna()
    ambas_preenchidas = d1_preenchida & d2_preenchida
    
    # REGRA DE MANUTENÇÃO: Mantém a linha se pelo menos UMA data preenchida estiver dentro do período
    manter_linha = (d1_preenchida & d1_in) | (d2_preenchida & d2_in)
    
    df_filtrado = df[manter_linha].copy()
    
    # Re-calcula máscaras no df_filtrado para atribuir a nova coluna corretamente
    d1_f = df_filtrado[col1] if col1 in df_filtrado.columns else pd.Series([pd.NaT]*len(df_filtrado), index=df_filtrado.index)
    d2_f = df_filtrado[col2] if col2 in df_filtrado.columns else pd.Series([pd.NaT]*len(df_filtrado), index=df_filtrado.index)
    
    d1_in_f = (d1_f >= data_ini) & (d1_f <= data_fim)
    d2_in_f = (d2_f >= data_ini) & (d2_f <= data_fim)
    ambas_preenchidas_f = d1_f.notna() & d2_f.notna()
    
    # CRIAÇÃO DA NOVA COLUNA
    # Padrão: NÃO
    df_filtrado["FOI BAIXADO ANTES"] = "NÃO"
    
    # Regra para SIM: Ambas colunas existem E (d1 está fora OU d2 está fora)
    # Como já filtramos garantindo que uma está dentro, basta verificar se NÃO estão as duas dentro
    condicao_sim = ambas_preenchidas_f & ~(d1_in_f & d2_in_f)
    
    df_filtrado.loc[condicao_sim, "FOI BAIXADO ANTES"] = "SIM"
    
    return df_filtrado

baixa_filtrado = filtrar_duas_datas(baixa, col_data_baixa, col_data_enc, ini, fim)
enc_filtrado = filtrar_duas_datas(enc, col_data_baixa, col_data_enc, ini, fim)


# ===============================
# AJUSTE DAS COLUNAS DA BAIXA
# ===============================

# para padronizar com ENCERRADOS
# Obs: Se 'baixa' já possuía a coluna "Data de Encerramento" e você não quiser 
# sobrescrevê-la, comente/remova a primeira linha abaixo.
baixa_filtrado[col_data_enc] = baixa_filtrado[col_data_baixa]
baixa_filtrado[col_motivo_enc] = baixa_filtrado[col_motivo_bp]


# ===============================
# CRIAÇÃO DO SETTLED
# ===============================

settled = pd.concat([baixa_filtrado, enc_filtrado], ignore_index=True)

settled.to_excel("SETTLED_MENSAL.xlsx", index=False)


# ===============================
# RELATÓRIO FINAL
# ===============================

print("\nETAPA 7 concluída ✔")

print(f"Período aplicado: {ini.date()} até {fim.date()}")
print(f"BAIXA PROVISÓRIA no período: {len(baixa_filtrado)}")
print(f"ENCERRADOS no período: {len(enc_filtrado)}")
print(f"Total no SETTLED: {len(settled)}")

qtd_excecao = (settled["FOI BAIXADO ANTES"] == "SIM").sum()
print(f"Processos c/ datas conflitantes marcados com 'SIM': {qtd_excecao}")

print("\nArquivo gerado: SETTLED_MENSAL.xlsx")

Carregando planilhas...
Convertendo datas...

Diagnóstico de datas
BAIXA (Data) válidas: 256
BAIXA (Encerramento) válidas: 0
ENCERRADOS (Data) válidas: 158
ENCERRADOS (Encerramento) válidas: 375

Aplicando filtro de período...

ETAPA 7 concluída ✔
Período aplicado: 2026-03-26 até 2026-04-25
BAIXA PROVISÓRIA no período: 40
ENCERRADOS no período: 76
Total no SETTLED: 116
Processos c/ datas conflitantes marcados com 'SIM': 22

Arquivo gerado: SETTLED_MENSAL.xlsx


ETAPA 11 - Settled Acumulado

In [13]:
import pandas as pd



print("Carregando planilhas...")

baixa = pd.read_excel("BAIXA_PROVISORIA.xlsx")
enc = pd.read_excel("ENCERRADOS.xlsx")

# remove espaços em nomes de colunas (muito comum no Excel)
baixa.columns = baixa.columns.str.strip()
enc.columns = enc.columns.str.strip()

# colunas utilizadas
col_data_baixa = "Data"
col_data_enc = "Data de Encerramento"

col_motivo_bp = "Motivo da Baixa Provisória"
col_motivo_enc = "Motivo Encerramento"


# ===============================
# FUNÇÃO ROBUSTA DE CONVERSÃO DE DATA
# ===============================

def conv_data(col):

    # tentativa padrão
    d = pd.to_datetime(col, dayfirst=True, errors="coerce")

    col_numeric = pd.to_numeric(col, errors="coerce")

    # trata números Excel (dias desde 1899)
    mask_excel = (
        d.isna() &
        col_numeric.notna() &
        (col_numeric > 0) &
        (col_numeric < 60000)
    )

    d.loc[mask_excel] = pd.to_datetime(
        col_numeric[mask_excel],
        unit="D",
        origin="1899-12-30"
    )

    # trata timestamp em nanossegundos
    mask_ns = (
        d.isna() &
        col_numeric.notna() &
        (col_numeric > 1e12)
    )

    d.loc[mask_ns] = pd.to_datetime(
        col_numeric[mask_ns],
        unit="ns"
    )

    return d


# ===============================
# CONVERSÃO DAS DATAS CORRETAS
# ===============================

print("Convertendo datas...")

# Função auxiliar para garantir que convertemos a coluna apenas se ela existir no df
def converter_se_existir(df, col):
    if col in df.columns:
        df[col] = conv_data(df[col])
        df[col] = df[col].dt.normalize() # já remove as horas

# Aplicando para a planilha BAIXA
converter_se_existir(baixa, col_data_baixa)
converter_se_existir(baixa, col_data_enc)

# Aplicando para a planilha ENCERRADOS
converter_se_existir(enc, col_data_baixa)
converter_se_existir(enc, col_data_enc)


# ===============================
# DIAGNÓSTICO DAS DATAS
# ===============================

print("\nDiagnóstico de datas")

if col_data_baixa in baixa.columns:
    print("BAIXA (Data) válidas:", baixa[col_data_baixa].notna().sum())
if col_data_enc in baixa.columns:
    print("BAIXA (Encerramento) válidas:", baixa[col_data_enc].notna().sum())

if col_data_baixa in enc.columns:
    print("ENCERRADOS (Data) válidas:", enc[col_data_baixa].notna().sum())
if col_data_enc in enc.columns:
    print("ENCERRADOS (Encerramento) válidas:", enc[col_data_enc].notna().sum())


# ===============================
# PERÍODO MANUAL
# ===============================

data_inicio_manual = "26/12/2025"
data_fim_manual = "25/04/2026"   # ALTERE AQUI

ini = pd.to_datetime(data_inicio_manual, dayfirst=True)
fim = pd.to_datetime(data_fim_manual, dayfirst=True)


# ===============================
# FILTROS CORRETOS (Com verificação e flag de exceção)
# ===============================

print("\nAplicando filtro de período...")

def filtrar_duas_datas(df, col1, col2, data_ini, data_fim):
    d1 = df[col1] if col1 in df.columns else pd.Series([pd.NaT]*len(df), index=df.index)
    d2 = df[col2] if col2 in df.columns else pd.Series([pd.NaT]*len(df), index=df.index)
    
    # Validações individuais (se a data está no período)
    d1_in = (d1 >= data_ini) & (d1 <= data_fim)
    d2_in = (d2 >= data_ini) & (d2 <= data_fim)
    
    # Verificação de quais estão preenchidas
    d1_preenchida = d1.notna()
    d2_preenchida = d2.notna()
    ambas_preenchidas = d1_preenchida & d2_preenchida
    
    # REGRA DE MANUTENÇÃO: Mantém a linha se pelo menos UMA data preenchida estiver dentro do período
    manter_linha = (d1_preenchida & d1_in) | (d2_preenchida & d2_in)
    
    df_filtrado = df[manter_linha].copy()
    
    # Re-calcula máscaras no df_filtrado para atribuir a nova coluna corretamente
    d1_f = df_filtrado[col1] if col1 in df_filtrado.columns else pd.Series([pd.NaT]*len(df_filtrado), index=df_filtrado.index)
    d2_f = df_filtrado[col2] if col2 in df_filtrado.columns else pd.Series([pd.NaT]*len(df_filtrado), index=df_filtrado.index)
    
    d1_in_f = (d1_f >= data_ini) & (d1_f <= data_fim)
    d2_in_f = (d2_f >= data_ini) & (d2_f <= data_fim)
    ambas_preenchidas_f = d1_f.notna() & d2_f.notna()
    
    # CRIAÇÃO DA NOVA COLUNA
    # Padrão: NÃO
    df_filtrado["FOI BAIXADO ANTES"] = "NÃO"
    
    # Regra para SIM: Ambas colunas existem E (d1 está fora OU d2 está fora)
    # Como já filtramos garantindo que uma está dentro, basta verificar se NÃO estão as duas dentro
    condicao_sim = ambas_preenchidas_f & ~(d1_in_f & d2_in_f)
    
    df_filtrado.loc[condicao_sim, "FOI BAIXADO ANTES"] = "SIM"
    
    return df_filtrado

baixa_filtrado = filtrar_duas_datas(baixa, col_data_baixa, col_data_enc, ini, fim)
enc_filtrado = filtrar_duas_datas(enc, col_data_baixa, col_data_enc, ini, fim)


# ===============================
# AJUSTE DAS COLUNAS DA BAIXA
# ===============================

# para padronizar com ENCERRADOS
# Obs: Se 'baixa' já possuía a coluna "Data de Encerramento" e você não quiser 
# sobrescrevê-la, comente/remova a primeira linha abaixo.
baixa_filtrado[col_data_enc] = baixa_filtrado[col_data_baixa]
baixa_filtrado[col_motivo_enc] = baixa_filtrado[col_motivo_bp]


# ===============================
# CRIAÇÃO DO SETTLED
# ===============================

settled = pd.concat([baixa_filtrado, enc_filtrado], ignore_index=True)

settled.to_excel("SETTLED_ACUMULADO.xlsx", index=False)


# ===============================
# RELATÓRIO FINAL
# ===============================

print("\nETAPA 7 concluída ✔")

print(f"Período aplicado: {ini.date()} até {fim.date()}")
print(f"BAIXA PROVISÓRIA no período: {len(baixa_filtrado)}")
print(f"ENCERRADOS no período: {len(enc_filtrado)}")
print(f"Total no SETTLED: {len(settled)}")

qtd_excecao = (settled["FOI BAIXADO ANTES"] == "SIM").sum()
print(f"Processos c/ datas conflitantes marcados com 'SIM': {qtd_excecao}")

print("\nArquivo gerado: SETTLED_ACUMULADO.xlsx")

Carregando planilhas...
Convertendo datas...

Diagnóstico de datas
BAIXA (Data) válidas: 256
BAIXA (Encerramento) válidas: 0
ENCERRADOS (Data) válidas: 158
ENCERRADOS (Encerramento) válidas: 375

Aplicando filtro de período...

ETAPA 7 concluída ✔
Período aplicado: 2025-12-26 até 2026-04-25
BAIXA PROVISÓRIA no período: 126
ENCERRADOS no período: 184
Total no SETTLED: 310
Processos c/ datas conflitantes marcados com 'SIM': 49

Arquivo gerado: SETTLED_ACUMULADO.xlsx


ETAPA 12 - Atualização do Histórico

In [14]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

col_pasta = "Pasta"
col_status = "Status"
col_data_cadastro = "Data de cadastro"

# =====================================================
# Ler SETTLED_MENSAL.xlsx (baixas e fechamentos do mês)
# =====================================================

if not os.path.exists("SETTLED_MENSAL.xlsx"):
    print("SETTLED_MENSAL.xlsx não existe. ETAPA 11 ignorada.")

else:

    try:
        novo = pd.read_excel("SETTLED_MENSAL.xlsx", dtype=str)
    except EmptyDataError:
        print("SETTLED_MENSAL.xlsx vazio. ETAPA 11 ignorada.")
        novo = pd.DataFrame()

    if novo.empty:
        print("Sem dados em SETTLED_MENSAL.")
    else:

        novo.columns = novo.columns.astype(str).str.strip()

        # =====================================================
        # Ler histórico
        # =====================================================

        hist = pd.DataFrame()

        if os.path.exists("SETTLED.xlsx"):

            try:
                hist = pd.read_excel("SETTLED.xlsx", dtype=str, engine="openpyxl")
            except Exception:
                print("SETTLED.xlsx vazio ou inválido. Criando histórico novo.")
                hist = pd.DataFrame(columns=novo.columns)

        else:
            hist = pd.DataFrame(columns=novo.columns)

        hist.columns = hist.columns.astype(str).str.strip()

        # =====================================================
        # Garantir colunas essenciais
        # =====================================================

        for col in [col_pasta, col_status, col_data_cadastro]:

            if col not in novo.columns:
                raise ValueError(f"Coluna obrigatória '{col}' não encontrada em SETTLED_2.xlsx")

            if col not in hist.columns:
                hist[col] = None

        # =====================================================
        # Padronizar campos
        # =====================================================

        novo[col_pasta] = novo[col_pasta].astype(str).str.strip()
        hist[col_pasta] = hist[col_pasta].astype(str).str.strip()

        novo[col_data_cadastro] = pd.to_datetime(novo[col_data_cadastro], errors="coerce")
        hist[col_data_cadastro] = pd.to_datetime(hist[col_data_cadastro], errors="coerce")

        # =====================================================
        # Remover duplicados
        # =====================================================

        novo = novo.drop_duplicates(subset=[col_pasta], keep="last")
        hist = hist.drop_duplicates(subset=[col_pasta], keep="last")

        # =====================================================
        # Merge histórico + novo
        # =====================================================

        merged = hist.merge(
            novo,
            on=col_pasta,
            how="outer",
            suffixes=("_hist", "_novo")
        )

        # =====================================================
        # Escolher registro mais recente (corrigir perda de linhas)
        # =====================================================
        novo_existe = (
            merged[f"{col_status}_novo"].notna() |
            merged[f"{col_data_cadastro}_novo"].notna()
        )

        cond_atualizar = novo_existe & (
            (merged[f"{col_status}_hist"] != merged[f"{col_status}_novo"]) |
            (merged[f"{col_data_cadastro}_novo"] > merged[f"{col_data_cadastro}_hist"])
        )

        # linhas que vêm do novo
        atualizar = merged[cond_atualizar]

        # linhas que mantêm histórico
        manter = merged[~cond_atualizar]

        # =====================================================
        # Reconstruir dataframe final (CORRIGIDO: incluir "Pasta")
        # =====================================================

        df_manter = manter[[c for c in merged.columns if c.endswith("_hist")]].rename(
            columns=lambda x: x.replace("_hist", "")
        ).copy()

        df_atualizar = atualizar[[c for c in merged.columns if c.endswith("_novo")]].rename(
            columns=lambda x: x.replace("_novo", "")
        ).copy()

        # Garantir que "Pasta" está presente em ambos os dataframes
        if col_pasta not in df_manter.columns and col_pasta in manter.columns:
            df_manter[col_pasta] = manter[col_pasta]

        if col_pasta not in df_atualizar.columns and col_pasta in atualizar.columns:
            df_atualizar[col_pasta] = atualizar[col_pasta]

        df_atualizado = pd.concat([df_manter, df_atualizar], ignore_index=True)
        df_atualizado = df_atualizado.reset_index(drop=True)

        # =====================================================
        # Salvar de forma segura
        # =====================================================

        temp_file = "SETTLED_temp.xlsx"
        df_atualizado.to_excel(temp_file, index=False)

        if os.path.exists("SETTLED.xlsx"):
            os.remove("SETTLED.xlsx")

        os.rename(temp_file, "SETTLED.xlsx")

        print("ETAPA 11 concluída ✔")
        print("Total histórico:", len(df_atualizado))
        print(f"Coluna 'Pasta' presente: {col_pasta in df_atualizado.columns}")

ETAPA 11 concluída ✔
Total histórico: 116
Coluna 'Pasta' presente: True


ETAPA 13 - Filtro BACENJUD

In [15]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

arquivo_settled = "SETTLED.xlsx"
arquivo_bacenjud = "BACENJUD.xlsx"

if not os.path.exists(arquivo_settled):
    print("SETTLED.xlsx não existe. Filtro BacenJud ignorado.")

else:
    try:
        settled = pd.read_excel(arquivo_settled)
    except EmptyDataError:
        print("SETTLED.xlsx está vazio. Filtro BacenJud ignorado.")
        settled = pd.DataFrame()

    if settled.empty:
        print("SETTLED sem dados. Filtro BacenJud ignorado.")

    else:

        col_garantia = "Garantia"

        if col_garantia in settled.columns:

            bacenjud = settled[
                settled[col_garantia] == "03 - Penhora Online (BacenJud)"
            ].copy()

            # salvar nova planilha
            bacenjud.to_excel(arquivo_bacenjud, index=False)

            print("Planilha BACENJUD criada.")
            print("Total de casos BacenJud:", len(bacenjud))

        else:
            print("Coluna Garantia não encontrada. Filtro BacenJud ignorado.")

Planilha BACENJUD criada.
Total de casos BacenJud: 1


ETAPA 14 - PAGAMENTOS 2.0

In [16]:
import pandas as pd

# =========================
# PARÂMETROS
# =========================
data_limite = pd.to_datetime("2026-04-25")

# =========================
# FUNÇÕES AUXILIARES
# =========================

def normalizar_pasta(col):
    """Remove problemas de formato na chave da pasta"""
    return (
        pd.to_numeric(col, errors="coerce")
        .astype("Int64")
        .astype(str)
        .str.strip()
    )

def limpar_valor_monetario(col):
    """Converte valores brasileiros para float sem distorcer"""
    if pd.api.types.is_numeric_dtype(col):
        return col.astype(float)

    return (
        col.astype(str)
        .str.replace("R$", "", regex=False)
        .str.replace(".", "", regex=False)   # remove milhar
        .str.replace(",", ".", regex=False)  # converte decimal
        .str.strip()
        .astype(float)
    )

# =========================
# CARREGAR PLANILHAS
# =========================

settled = pd.read_excel("SETTLED.xlsx")
pagamentos = pd.read_excel("Pagamentos.xlsx")

# =========================
# LIMPEZA DE COLUNAS
# =========================

pagamentos.columns = pagamentos.columns.str.strip().str.replace(r"\s+", " ", regex=True)
settled.columns = settled.columns.str.strip().str.replace(r"\s+", " ", regex=True)

# =========================
# NORMALIZAÇÃO DAS CHAVES
# =========================

settled["Pasta"] = normalizar_pasta(settled["Pasta"])
pagamentos["Nº Pasta Projuris"] = normalizar_pasta(pagamentos["Nº Pasta Projuris"])

print(
    "Pastas em comum:",
    len(set(settled["Pasta"]).intersection(set(pagamentos["Nº Pasta Projuris"])))
)

# =========================
# TRATAMENTO DE DATAS
# =========================

pagamentos["Data de Lançamento"] = pd.to_datetime(
    pagamentos["Data de Lançamento"],
    dayfirst=True,
    errors="coerce"
)

# =========================
# TRATAMENTO DE VALORES
# =========================

pagamentos["Valor do Lançamento"] = limpar_valor_monetario(
    pagamentos["Valor do Lançamento"]
)

# =========================
# FILTRO DE DATAS
# =========================

pag_filtrado = pagamentos[
    (pagamentos["Data de Lançamento"] > pd.Timestamp("2025-08-25")) &
    (pagamentos["Data de Lançamento"] < data_limite)
]

print("Linhas após filtro:", len(pag_filtrado))

# =========================
# SOMA DOS PAGAMENTOS
# =========================

soma_pagamentos = (
    pag_filtrado
    .groupby("Nº Pasta Projuris")["Valor do Lançamento"]
    .sum()
)

# =========================
# MAPEAR PARA SETTLED
# =========================

settled["Soma_Valor_Lancamento"] = settled["Pasta"].map(soma_pagamentos)

settled["Soma_Valor_Lancamento"] = settled["Soma_Valor_Lancamento"].fillna(0).round(2)

# =========================
# SALVAR RESULTADO
# =========================

settled.to_excel("SETTLED.xlsx", index=False)

print("Linhas SETTLED:", len(settled))
print("Pastas com pagamento:", settled["Soma_Valor_Lancamento"].gt(0).sum())

Pastas em comum: 40
Linhas após filtro: 589
Linhas SETTLED: 116
Pastas com pagamento: 25


ETAPA 14.2 - Pagamentos para SETTLED ACUMULADO

In [17]:
import pandas as pd

# =========================
# PARÂMETROS
# =========================
data_limite = pd.to_datetime("2026-04-25")

# =========================
# FUNÇÕES AUXILIARES
# =========================

def normalizar_pasta(col):
    """Remove problemas de formato na chave da pasta"""
    return (
        pd.to_numeric(col, errors="coerce")
        .astype("Int64")
        .astype(str)
        .str.strip()
    )

def limpar_valor_monetario(col):
    """Converte valores brasileiros para float sem distorcer"""
    if pd.api.types.is_numeric_dtype(col):
        return col.astype(float)

    return (
        col.astype(str)
        .str.replace("R$", "", regex=False)
        .str.replace(".", "", regex=False)   # remove milhar
        .str.replace(",", ".", regex=False)  # converte decimal
        .str.strip()
        .astype(float)
    )

# =========================
# CARREGAR PLANILHAS
# =========================

settled = pd.read_excel("SETTLED_ACUMULADO.xlsx")
pagamentos = pd.read_excel("Pagamentos.xlsx")

# =========================
# LIMPEZA DE COLUNAS
# =========================

pagamentos.columns = pagamentos.columns.str.strip().str.replace(r"\s+", " ", regex=True)
settled.columns = settled.columns.str.strip().str.replace(r"\s+", " ", regex=True)

# =========================
# NORMALIZAÇÃO DAS CHAVES
# =========================

settled["Pasta"] = normalizar_pasta(settled["Pasta"])
pagamentos["Nº Pasta Projuris"] = normalizar_pasta(pagamentos["Nº Pasta Projuris"])

print(
    "Pastas em comum:",
    len(set(settled["Pasta"]).intersection(set(pagamentos["Nº Pasta Projuris"])))
)

# =========================
# TRATAMENTO DE DATAS
# =========================

pagamentos["Data de Lançamento"] = pd.to_datetime(
    pagamentos["Data de Lançamento"],
    dayfirst=True,
    errors="coerce"
)

# =========================
# TRATAMENTO DE VALORES
# =========================

pagamentos["Valor do Lançamento"] = limpar_valor_monetario(
    pagamentos["Valor do Lançamento"]
)

# =========================
# FILTRO DE DATAS
# =========================

pag_filtrado = pagamentos[
    (pagamentos["Data de Lançamento"] > pd.Timestamp("2025-08-25")) &
    (pagamentos["Data de Lançamento"] < data_limite)
]

print("Linhas após filtro:", len(pag_filtrado))

# =========================
# SOMA DOS PAGAMENTOS
# =========================

soma_pagamentos = (
    pag_filtrado
    .groupby("Nº Pasta Projuris")["Valor do Lançamento"]
    .sum()
)

# =========================
# MAPEAR PARA SETTLED
# =========================

settled["Soma_Valor_Lancamento"] = settled["Pasta"].map(soma_pagamentos)

settled["Soma_Valor_Lancamento"] = settled["Soma_Valor_Lancamento"].fillna(0).round(2)

# =========================
# SALVAR RESULTADO
# =========================

settled.to_excel("SETTLED_ACUMULADO.xlsx", index=False)

print("Linhas SETTLED:", len(settled))
print("Pastas com pagamento:", settled["Soma_Valor_Lancamento"].gt(0).sum())

Pastas em comum: 139
Linhas após filtro: 589
Linhas SETTLED: 310
Pastas com pagamento: 104


ETAPA 14.3 - Pagamentos para SETTLED_MENSAL

In [18]:
import pandas as pd

# =========================
# PARÂMETROS
# =========================
data_limite = pd.to_datetime("2026-04-25")

# =========================
# FUNÇÕES AUXILIARES
# =========================

def normalizar_pasta(col):
    """Remove problemas de formato na chave da pasta"""
    return (
        pd.to_numeric(col, errors="coerce")
        .astype("Int64")
        .astype(str)
        .str.strip()
    )

def limpar_valor_monetario(col):
    """Converte valores brasileiros para float sem distorcer"""
    if pd.api.types.is_numeric_dtype(col):
        return col.astype(float)

    return (
        col.astype(str)
        .str.replace("R$", "", regex=False)
        .str.replace(".", "", regex=False)   # remove milhar
        .str.replace(",", ".", regex=False)  # converte decimal
        .str.strip()
        .astype(float)
    )

# =========================
# CARREGAR PLANILHAS
# =========================

settled = pd.read_excel("SETTLED_MENSAL.xlsx")
pagamentos = pd.read_excel("Pagamentos.xlsx")

# =========================
# LIMPEZA DE COLUNAS
# =========================

pagamentos.columns = pagamentos.columns.str.strip().str.replace(r"\s+", " ", regex=True)
settled.columns = settled.columns.str.strip().str.replace(r"\s+", " ", regex=True)

# =========================
# NORMALIZAÇÃO DAS CHAVES
# =========================

settled["Pasta"] = normalizar_pasta(settled["Pasta"])
pagamentos["Nº Pasta Projuris"] = normalizar_pasta(pagamentos["Nº Pasta Projuris"])

print(
    "Pastas em comum:",
    len(set(settled["Pasta"]).intersection(set(pagamentos["Nº Pasta Projuris"])))
)

# =========================
# TRATAMENTO DE DATAS
# =========================

pagamentos["Data de Lançamento"] = pd.to_datetime(
    pagamentos["Data de Lançamento"],
    dayfirst=True,
    errors="coerce"
)

# =========================
# TRATAMENTO DE VALORES
# =========================

pagamentos["Valor do Lançamento"] = limpar_valor_monetario(
    pagamentos["Valor do Lançamento"]
)

# =========================
# FILTRO DE DATAS
# =========================

pag_filtrado = pagamentos[
    (pagamentos["Data de Lançamento"] > pd.Timestamp("2025-08-25")) &
    (pagamentos["Data de Lançamento"] < data_limite)
]

print("Linhas após filtro:", len(pag_filtrado))

# =========================
# SOMA DOS PAGAMENTOS
# =========================

soma_pagamentos = (
    pag_filtrado
    .groupby("Nº Pasta Projuris")["Valor do Lançamento"]
    .sum()
)

# =========================
# MAPEAR PARA SETTLED
# =========================

settled["Soma_Valor_Lancamento"] = settled["Pasta"].map(soma_pagamentos)

settled["Soma_Valor_Lancamento"] = settled["Soma_Valor_Lancamento"].fillna(0).round(2)

# =========================
# SALVAR RESULTADO
# =========================

settled.to_excel("SETTLED_MENSAL.xlsx", index=False)

print("Linhas SETTLED:", len(settled))
print("Pastas com pagamento:", settled["Soma_Valor_Lancamento"].gt(0).sum())

Pastas em comum: 40
Linhas após filtro: 589
Linhas SETTLED: 116
Pastas com pagamento: 25


ETAPA 15 - Macro Assunto

In [19]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

# =========================
# mapa de classificação (Assunto)
# =========================

mapa_macro = {
    "FAR CEF - Vício Construtivo Área Comum": "Construction",
    "Reclamação Proprietário - Vício Construtivo Área Comum": "Construction",
    "Vício Construtivo - Área Comum": "Construction",
    "Atraso de Obra": "Delay",
    "Entrega do Imóvel": "Delay",
    "FAR - Vício Construtivo Unidade Autônoma": "FAR",
    "FAR CEF - Vício Construtivo Unidade Autônoma": "FAR",
    "Acidente de Trabalho": "Labor",
    "Dano Material l Moral": "Cível",
    "Esclarecimento de Informações": "Cível",
    "Horas Extras": "Labor",
    "Reclamação Trabalhista": "Labor",
    "Responsabilidade Solidária/Subsidiária": "Labor",
    "Alienação Fiduciária": "Cível",
    "Ambiental": "Tax",
    "Anulação de Cobrança": "Cível",
    "Arbitragem": "Cível",
    "Cobrança": "Tax",
    "Comissão de Corretagem": "Cível",
    "Corrupção": "Cível",
    "Cota Condominial": "Cível",
    "Desapropriação": "Cível",
    "Descumprimento da Oferta": "Cível",
    "DF Century - Desocupação": "Cível",
    "Embargo de Obra": "Cível",
    "Entrega de Documentos": "Cível",
    "Fornecedor": "Cível",
    "Hipoteca": "Cível",
    "Honorários Advocatícios": "Cível",
    "Incapacidade Financeira": "Cível",
    "Inexistência de Débito": "Cível",
    "Laudêmio": "Tax",
    "Leilão": "Cível",
    "Licenças Públicas": "Cível",
    "Multa Procon": "Tax",
    "Negativação Indevida": "Cível",
    "Obrigação de Fazer/Não Fazer": "Cível",
    "Outorga de Escritura": "Cível",
    "Parceria": "Cível",
    "Percentual Retido Distrato": "Cível",
    "Permutante": "Cível",
    "Posse": "Cível",
    "Protesto - Serviços (água, gás, energia, esgoto)": "Cível",
    "Relação de Consumo": "Cível",
    "Repasse": "Cível",
    "Rescisão Contratual": "Cível",
    "Rescisão Contratual - Locação": "Cível",
    "Restituição de Valores": "Cível",
    "Vizinho": "Cível",
    "Vício Construtivo - Unidade Autônoma": "Cível",
    "IPTU Cliente": "Property Tax",
    "Protesto - IPTU": "Property Tax",
    "CIDE": "Tax",
    "Contribuição Previdenciária": "Tax",
    "CREA": "Tax",
    "CSLL": "Tax",
    "Execução": "Tax",
    "ICMS": "Tax",
    "IE": "Tax",
    "IOF": "Tax",
    "IRPJ": "Tax",
    "ISS": "Tax",
    "ITBI": "Tax",
    "PerDcomp": "Tax",
    "PIS/COFINS": "Tax",
    "Preço Público": "Tax",
    "Protesto": "Tax",
    "Taxa": "Tax",
    "Taxa de Lixo": "Tax",
    "IPTU": "Tax",
    "Castelos D'água": "Cível"
}

# =========================
# mapa de fallback (Natureza)
# =========================

mapa_natureza = {
    "Cível - Estratégico": "Cível",
    "Administrativo - Tributário": "Tax",
    "Tributário": "Tax",
    "Cível": "Cível",
    "Trabalhista": "Labor",
    "Administrativo - Estratégico": "Cível",
    "Trabalhista - Estratégico": "Labor",
    "Administrativo": "Cível",
    "Administrativo - Trabalhista": "Labor"
}

# =========================
# arquivos
# =========================

arquivos = [
    "SETTLED.xlsx",
    "SETTLED_MENSAL.xlsx",
    "POS_BP.xlsx",
    "ENTRADAS.xlsx",
    "SETTLED_ACUMULADO.xlsx"
]

col_origem = "Assunto Principal"
col_destino = "Macro Assunto"

# =========================
# processamento
# =========================

for arquivo in arquivos:

    if not os.path.exists(arquivo):
        print(f"{arquivo} não existe. Tradução ignorada.")
        continue

    try:
        df = pd.read_excel(arquivo)
    except EmptyDataError:
        print(f"{arquivo} está vazio. Tradução ignorada.")
        continue

    if df.empty:
        print(f"{arquivo} sem dados. Tradução ignorada.")
        continue

    # limpa nomes das colunas
    df.columns = df.columns.str.strip()

    if col_origem not in df.columns:
        print(f"Coluna '{col_origem}' não encontrada em {arquivo}. Tradução ignorada.")
        continue

    # limpa campos
    df[col_origem] = df[col_origem].astype(str).str.strip()

    if "Natureza" in df.columns:
        df["Natureza"] = df["Natureza"].astype(str).str.strip()

    # cria coluna destino se não existir
    if col_destino not in df.columns:
        df[col_destino] = ""

    df[col_destino] = df[col_destino].fillna("").astype(str)

    # =========================
    # 1. MAPEAMENTO POR ASSUNTO
    # =========================

    mask_vazio = df[col_destino].str.strip() == ""

    df.loc[mask_vazio, col_destino] = df.loc[mask_vazio, col_origem].map(mapa_macro)

    # =========================
    # 2. FALLBACK POR NATUREZA
    # =========================

    mask_restante = df[col_destino].isna() | (df[col_destino].str.strip() == "")

    if "Natureza" in df.columns:
        df.loc[mask_restante, col_destino] = df.loc[mask_restante, "Natureza"].map(mapa_natureza)

    # =========================
    # 3. LOG DE NÃO CLASSIFICADOS
    # =========================

    nao_classificados = df[
        df[col_destino].isna() | (df[col_destino].str.strip() == "")
    ][col_origem].dropna().unique()

    if len(nao_classificados) > 0:
        print(f"\nAssuntos sem classificação em {arquivo}:")
        print(sorted(nao_classificados))

    # =========================
    # SALVAR
    # =========================

    df.to_excel(arquivo, index=False)

    print(f"Macro Assunto atualizado em {arquivo}")

Macro Assunto atualizado em SETTLED.xlsx
Macro Assunto atualizado em SETTLED_MENSAL.xlsx
Macro Assunto atualizado em POS_BP.xlsx
Macro Assunto atualizado em ENTRADAS.xlsx
Macro Assunto atualizado em SETTLED_ACUMULADO.xlsx


ETAPA 16 - Macro Encerramento

In [20]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

def processar_settled(nome_arquivo):

    if not os.path.exists(nome_arquivo):
        print(f"{nome_arquivo} não existe. Macro encerramento ignorado.")
        return

    try:
        df = pd.read_excel(nome_arquivo)
    except EmptyDataError:
        print(f"{nome_arquivo} está vazio. Macro encerramento ignorado.")
        return

    if df.empty:
        print(f"{nome_arquivo} sem dados. Macro encerramento ignorado.")
        return

    col_macro = "Macro encerramento"
    col_motivo = "Motivo Encerramento"
    col_pasta = "Pasta"
    col_pagamento = "Soma_Valor_Lancamento"

    if col_motivo in df.columns and col_pasta in df.columns:

        # criar coluna se não existir
        if col_macro not in df.columns:
            df[col_macro] = ""

        mask_vazio = df[col_macro].isna() | (df[col_macro].astype(str).str.strip() == "")

        df_vazio = df[mask_vazio]
        df_vazio.to_excel(f"Relatorio_Macroencerramento_{nome_arquivo}", index=False)

        valores_settled = [
            "Acordo",
            "Acordo pago - Aguardando baixa",
            "Acordo parcelado- Aguardando baixa",
            "REFIS/PPI - Aguardando baixa"
        ]

        mask_settled = (
            mask_vazio &
            df[col_motivo].isin(valores_settled) &
            ~df[col_motivo].str.contains("terceiro", case=False, na=False)
        )

        df.loc[mask_settled, col_macro] = "Settled"

        valores_lost = [
            "Condenação paga - Aguardando baixa",
            "Procedência"
        ]

        mask_lost = (
            mask_vazio &
            df[col_motivo].isin(valores_lost) &
            ~df[col_motivo].str.contains("terceiro", case=False, na=False)
        )

        df.loc[mask_lost, col_macro] = "Lost"

        if col_pagamento in df.columns:

            mask_restante = (
                df[col_macro].isna() |
                (df[col_macro].astype(str).str.strip() == "")
            )

            mask_won = (
                mask_restante &
                (df[col_pagamento]==0)
            )

            df.loc[mask_won, col_macro] = "Won"

        sobras = df[
            df[col_macro].isna() |
            (df[col_macro].astype(str).str.strip() == "")
        ]

        if len(sobras) > 0:
            print(f"Arquivos incorretos em {nome_arquivo}:")
            print(sobras[col_pasta].tolist())

        df.to_excel(nome_arquivo, index=False)

        print(f"{nome_arquivo} atualizado com Macro encerramento.")

    else:
        print(f"Colunas necessárias não encontradas em {nome_arquivo}.")


# ==========================================================
# RODAR PARA OS DOIS ARQUIVOS
# ==========================================================

processar_settled("SETTLED.xlsx")
processar_settled("SETTLED_MENSAL.xlsx")
processar_settled("SETTLED_ACUMULADO.xlsx")


Arquivos incorretos em SETTLED.xlsx:
[5750, 62893, 66667, 66997, 67788, 68527]
SETTLED.xlsx atualizado com Macro encerramento.
Arquivos incorretos em SETTLED_MENSAL.xlsx:
[68527, 66997, 66667, 67788, 62893, 5750]
SETTLED_MENSAL.xlsx atualizado com Macro encerramento.
Arquivos incorretos em SETTLED_ACUMULADO.xlsx:
[68527, 53385, 14354, 74148, 36806, 36776, 21576, 25551, 9901, 9751, 7949, 66997, 66667, 67788, 62893, 5750]
SETTLED_ACUMULADO.xlsx atualizado com Macro encerramento.


ETAPA 17 - Assumptions

In [41]:
import pandas as pd
import os

print("===== GERANDO ASSUMPTIONS_26 =====")

# ==============================
# VALORES FIXOS
# ==============================

# ALTERE AQUI caso os valores mudem no futuro
EXPECTED_LOSS_BASE = 597000000
EXPECTED_LOSS_FIXO = 54400000

# ALTERE AQUI caso o valor fixo do subtotal mude
SUBTOTAL_FIXO = 54400000


# ==============================
# PROCESSOS SEM ATUALIZAÇÃO
# ==============================

sem_atualizacao = 0


df = pd.read_excel("ATIVOS_SEM_VARIACAO.xlsx")

AY = df["Valor Pedido Objeto Corrigido"]

sem_atualizacao = AY.sum()




# ==============================
# PROCESSOS COM ATUALIZAÇÃO
# ==============================

com_atualizacao = 0

if os.path.exists("ATIVOS_COM_VARIACAO.xlsx"):

    ativos_var = pd.read_excel("ATIVOS_COM_VARIACAO.xlsx")

    AY = ativos_var["Valor Pedido Objeto Corrigido"]

    com_atualizacao = AY.sum()

else:
    print("ATIVOS_COM_VARIACAO.xlsx não encontrado")


# ==============================
# SETTLED
# ==============================

finally_resolved = 0
savings = 0

if os.path.exists("SETTLED.xlsx"):

    settled = pd.read_excel("SETTLED.xlsx")

    AY = settled["Valor Pedido Objeto Corrigido"]
    acordo = settled["Soma_Valor_Lancamento"]

    
    acordo = pd.to_numeric(acordo, errors="coerce")

    finally_resolved = acordo.sum()

    savings = AY.sum() - acordo.sum()

else:
    print("SETTLED.xlsx não encontrado")


# ==============================
# MUDANÇA DE PROGNÓSTICO
# ==============================

mudanca = 0

if os.path.exists("MUDANCA_DE_PROGNOSTICO.xlsx"):

    mud = pd.read_excel("MUDANCA_DE_PROGNOSTICO.xlsx")

    AY = mud["Valor Pedido Objeto Corrigido"]

    mudanca = AY.sum()

else:
    print("MUDANCA_DE_PROGNOSTICO.xlsx não encontrado")


# ==============================
# NEW CLAIMS
# ==============================

new_claims = 0

if os.path.exists("ENTRADAS.xlsx"):

    entradas = pd.read_excel("ENTRADAS.xlsx")

    AZ = entradas["Valor Pedido.1"]

    ativos = entradas["Status"].astype(str).str.upper() == "ATIVOS"

    new_claims = AZ[ativos].sum()

else:
    print("ENTRADAS.xlsx não encontrado")


# ==============================
# EXPECTED LOSSES
# ==============================

expected_losses_col1 = EXPECTED_LOSS_BASE
expected_losses_col2 = EXPECTED_LOSS_FIXO
expected_losses_col3 = expected_losses_col1 + expected_losses_col2


# ==============================
# SUBTOTAL
# ==============================

subtotal_col1 = (
    expected_losses_col1
    + mudanca
    - finally_resolved
    - savings
)

subtotal_col2 = SUBTOTAL_FIXO
subtotal_col3 = subtotal_col1 + subtotal_col2


# ==============================
# EXPECTED LOSSES MENSAL
# ==============================

expected_losses_mensal = subtotal_col1 + new_claims


# ==============================
# DATAFRAME FINAL
# ==============================

assumptions = pd.DataFrame({

    "Descrição": [
        "Processos sem atualização financeira",
        "Processos com atualização financeira ativos",
        "Expected Losses",
        "Finally resolved claims",
        "Savings from finally resolved claims",
        "Changes in expected losses",
        "Subtotal",
        "New Claims",
        "Expected Losses (Mensal)"
    ],

    "Calculo": [
        sem_atualizacao,
        com_atualizacao,
        expected_losses_col1,
        finally_resolved,
        savings,
        mudanca,
        subtotal_col1,
        new_claims,
        expected_losses_mensal
    ],

    "Fixo": [
        "",
        "",
        expected_losses_col2,
        "",
        "",
        "",
        subtotal_col2,
        "",
        ""
    ],

    "Soma": [
        "",
        "",
        expected_losses_col3,
        "",
        "",
        "",
        subtotal_col3,
        "",
        ""
    ]
})


# ==============================
# AJUSTE DE ESCALA (DIVIDIR POR 1000)
# ==============================

linhas_dividir = [
    "Processos sem atualização financeira",
    "Processos com atualização financeira ativos",
    "Finally resolved claims",
    "Savings from finally resolved claims",
    "Changes in expected losses",
    "New Claims",
    "Expected Losses (Mensal)"
]

mask = assumptions["Descrição"].isin(linhas_dividir)




# ==============================
# SALVAR EXCEL
# ==============================

assumptions.to_excel("assumptions_26.xlsx", index=False)

print("assumptions_26.xlsx criado com sucesso")

===== GERANDO ASSUMPTIONS_26 =====
assumptions_26.xlsx criado com sucesso


ETAPA 18 - Assumptions para o slide 1 

In [2]:
import pandas as pd
import os

print("===== GERANDO ASSUMPTIONS_26 =====")

# ==============================
# VALORES FIXOS
# ==============================

# ALTERE AQUI caso os valores mudem no futuro
EXPECTED_LOSS_BASE = 566000000   
EXPECTED_LOSS_FIXO = 54400000

# ALTERE AQUI caso o valor fixo do subtotal mude
SUBTOTAL_FIXO = 54400000


# ==============================
# PROCESSOS SEM ATUALIZAÇÃO
# ==============================

sem_atualizacao = 0


df = pd.read_excel("ATIVOS_SEM_VARIACAO.xlsx")

AY = df["Valor Pedido Objeto Corrigido"]

sem_atualizacao = AY.sum()




# ==============================
# PROCESSOS COM ATUALIZAÇÃO
# ==============================

com_atualizacao = 0

if os.path.exists("ATIVOS_COM_VARIACAO.xlsx"):

    ativos_var = pd.read_excel("ATIVOS_COM_VARIACAO.xlsx")

    AY = ativos_var["Valor Pedido Objeto Corrigido"]

    com_atualizacao = AY.sum()

else:
    print("ATIVOS_COM_VARIACAO.xlsx não encontrado")

# ==============================
# SETTLED
# ==============================
import os
import pandas as pd

finally_resolved = 0
savings = 0

if os.path.exists("SETTLED_ACUMULADO.xlsx"):

    settled = pd.read_excel("SETTLED_ACUMULADO.xlsx")

    
    # Isola as colunas e garante que são números para evitar erros de soma
    AY = pd.to_numeric(settled["Valor Pedido Objeto Corrigido"], errors="coerce")
    acordo = pd.to_numeric(settled["Soma_Valor_Lancamento"], errors="coerce")

    # Cálculos
    finally_resolved = acordo.sum()
    savings = AY.sum() - acordo.sum()

else:
    print("SETTLED_ACUMULADO.xlsx não encontrado")


# ==============================
# MUDANÇA DE PROGNÓSTICO
# ==============================

mudanca = 0

if os.path.exists("MUDANCA_DE_PROGNOSTICO.xlsx"):

    mud = pd.read_excel("MUDANCA_DE_PROGNOSTICO.xlsx")

    AY = mud["Valor Pedido Objeto Corrigido"]

    mudanca = AY.sum()

else:
    print("MUDANCA_DE_PROGNOSTICO.xlsx não encontrado")

# ==============================
# NEW CLAIMS
# ==============================

new_claims = 0

if os.path.exists("POS_BP.xlsx"):

    entradas = pd.read_excel("POS_BP.xlsx")

    AZ = entradas["Valor Pedido.1"]


    new_claims = AZ.sum()

else:
    print("POS_BP.xlsx não encontrado")

    
# ==============================
# EXPECTED LOSSES
# ==============================

expected_losses_col1 = EXPECTED_LOSS_BASE
expected_losses_col2 = EXPECTED_LOSS_FIXO
expected_losses_col3 = expected_losses_col1 + expected_losses_col2


# ==============================
# SUBTOTAL
# ==============================

subtotal_col1 = (
    expected_losses_col1
    + mudanca
    - finally_resolved
    - savings
)

subtotal_col2 = SUBTOTAL_FIXO
subtotal_col3 = subtotal_col1 + subtotal_col2


# ==============================
# EXPECTED LOSSES MENSAL
# ==============================

expected_losses_mensal = subtotal_col1 + new_claims


# ==============================
# DATAFRAME FINAL
# ==============================

assumptions = pd.DataFrame({

    "Descrição": [
        "Processos sem atualização financeira",
        "Processos com atualização financeira ativos",
        "Expected Losses",
        "Finally resolved claims",
        "Savings from finally resolved claims",
        "Changes in expected losses",
        "Subtotal",
        "New Claims",
        "Expected Losses (Mensal)"
    ],

    "Calculo": [
        sem_atualizacao,
        com_atualizacao,
        expected_losses_col1,
        finally_resolved,
        savings,
        mudanca,
        subtotal_col1,
        new_claims,
        expected_losses_mensal
    ],

    "Fixo": [
        "",
        "",
        expected_losses_col2,
        "",
        "",
        "",
        subtotal_col2,
        "",
        ""
    ],

    "Soma": [
        "",
        "",
        expected_losses_col3,
        "",
        "",
        "",
        subtotal_col3,
        "",
        ""
    ]
})


# ==============================
# AJUSTE DE ESCALA (DIVIDIR POR 1000)
# ==============================

linhas_dividir = [
    "Processos sem atualização financeira",
    "Processos com atualização financeira ativos",
    "Finally resolved claims",
    "Savings from finally resolved claims",
    "Changes in expected losses",
    "New Claims",
    "Expected Losses (Mensal)"
]

mask = assumptions["Descrição"].isin(linhas_dividir)




# ==============================
# SALVAR EXCEL
# ==============================

assumptions.to_excel("assumptions_26_slides.xlsx", index=False)

print("assumptions_26_slides.xlsx criado com sucesso")

===== GERANDO ASSUMPTIONS_26 =====
assumptions_26_slides.xlsx criado com sucesso


ETAPA 19 - Juntar Ativos (com Macro Assunto) e Entradas do Mês

In [23]:
import pandas as pd

df1 = pd.read_excel("ATIVOS_COM_VARIACAO.xlsx")
df2 = pd.read_excel("ATIVOS_SEM_VARIACAO.xlsx")
df3 = pd.read_excel("MUDANCA_DE_PROGNOSTICO.xlsx")

df_at = pd.concat([df1, df2, df3], ignore_index=True)

df_at.to_excel("ATIVOS_UNIFICADA.xlsx", index=False)

In [24]:
import os
import pandas as pd
from pandas.errors import EmptyDataError

# =========================
# mapa de classificação
# =========================

mapa_macro = {

    "FAR CEF - Vício Construtivo Área Comum": "Construction",
    "Reclamação Proprietário - Vício Construtivo Área Comum": "Construction",
    "Vício Construtivo - Área Comum": "Construction",
    "Atraso de Obra": "Delay",
    "Entrega do Imóvel": "Delay",
    "FAR - Vício Construtivo Unidade Autônoma": "FAR",
    "FAR CEF - Vício Construtivo Unidade Autônoma": "FAR",
    "Acidente de Trabalho": "Labor",
    "Dano Material l Moral": "Cível",
    "Esclarecimento de Informações": "Cível",
    "Horas Extras": "Labor",
    "Reclamação Trabalhista": "Labor",
    "Responsabilidade Solidária/Subsidiária": "Labor",
    "Alienação Fiduciária": "Cível",
    "Ambiental": "Tax",
    "Anulação de Cobrança": "Cível",
    "Arbitragem": "Cível",
    "Cobrança": "Tax",
    "Comissão de Corretagem": "Cível",
    "Corrupção": "Cível",
    "Cota Condominial": "Cível",
    "Desapropriação": "Cível",
    "Descumprimento da Oferta": "Cível",
    "DF Century - Desocupação": "Cível",
    "Embargo de Obra": "Cível",
    "Entrega de Documentos": "Cível",
    "Fornecedor": "Cível",
    "Hipoteca": "Cível",
    "Honorários Advocatícios": "Cível",
    "Incapacidade Financeira": "Cível",
    "Inexistência de Débito": "Cível",
    "Laudêmio": "Tax",
    "Leilão": "Cível",
    "Licenças Públicas": "Cível",
    "Multa Procon": "Tax",
    "Negativação Indevida": "Cível",
    "Obrigação de Fazer/Não Fazer": "Cível",
    "Outorga de Escritura": "Cível",
    "Parceria": "Cível",
    "Percentual Retido Distrato": "Cível",
    "Permutante": "Cível",
    "Posse": "Cível",
    "Protesto - Serviços (água, gás, energia, esgoto)": "Cível",
    "Relação de Consumo": "Cível",
    "Repasse": "Cível",
    "Rescisão Contratual": "Cível",
    "Rescisão Contratual - Locação": "Cível",
    "Restituição de Valores": "Cível",
    "Vizinho": "Cível",
    "Vício Construtivo - Unidade Autônoma": "Cível",
    "IPTU Cliente": "Property Tax",
    "Protesto - IPTU": "Property Tax",
    "CIDE": "Tax",
    "Contribuição Previdenciária": "Tax",
    "CREA": "Tax",
    "CSLL": "Tax",
    "Execução": "Tax",
    "ICMS": "Tax",
    "IE": "Tax",
    "IOF": "Tax",
    "IRPJ": "Tax",
    "ISS": "Tax",
    "ITBI": "Tax",
    "PerDcomp": "Tax",
    "PIS/COFINS": "Tax",
    "Preço Público": "Tax",
    "Protesto": "Tax",
    "Taxa": "Tax",
    "Taxa de Lixo": "Tax",
    "IPTU": "Tax",
    "Castelos D'água": "Cível"
}

# =========================
# arquivos para aplicar
# =========================

arquivos = [
"ATIVOS_UNIFICADA.xlsx"
]

col_origem = "Assunto Principal"
col_destino = "Macro Assunto"

# =========================
# loop nos arquivos
# =========================

for arquivo in arquivos:

    if not os.path.exists(arquivo):
        print(f"{arquivo} não existe. Tradução ignorada.")
        continue

    try:
        df = pd.read_excel(arquivo)
    except EmptyDataError:
        print(f"{arquivo} está vazio. Tradução ignorada.")
        continue

    if df.empty:
        print(f"{arquivo} sem dados. Tradução ignorada.")
        continue

    df.columns = df.columns.str.strip()

    if col_origem not in df.columns:
        print(f"Coluna '{col_origem}' não encontrada em {arquivo}. Tradução ignorada.")
        continue

    # limpar assunto principal
    df[col_origem] = df[col_origem].astype(str).str.strip()

    # criar coluna Macro Assunto se não existir
    if col_destino not in df.columns:
        df[col_destino] = ""

    # identificar assuntos não classificados
    nao_classificados = df[df[col_origem].map(mapa_macro).isna()][col_origem].dropna().unique()

    if len(nao_classificados) > 0:
        print(f"\nAssuntos sem classificação em {arquivo}:")
        print(sorted(nao_classificados))

    # preencher apenas vazios
    df[col_destino] = df[col_destino].fillna("").astype(str)
    mask = df[col_destino].str.strip() == ""

    df.loc[mask, col_destino] = df.loc[mask, col_origem].map(mapa_macro)

    # salvar
    df.to_excel(arquivo, index=False)

    print(f"Macro Assunto atualizado em {arquivo}")


Assuntos sem classificação em ATIVOS_UNIFICADA.xlsx:
['Cláusulas Promessa Compra e Venda', 'Cobrança de Tributo Federal', 'Descumprimento de NR', 'FAR - Vício Construtivo Área Comum', 'INSS', 'Procedimento Administrativo', 'Verba Rescisória', 'Vínculo Empregatício']
Macro Assunto atualizado em ATIVOS_UNIFICADA.xlsx


In [25]:
import pandas as pd

# carregar os arquivos
df1 = pd.read_excel("ATIVOS_UNIFICADA.xlsx")
df2 = pd.read_excel("ENTRADAS.xlsx")


# juntar as bases
df_final = pd.concat([df1, df2], ignore_index=True)

# salvar
df_final.to_excel("BASE_UNIFICADA.xlsx", index=False)

# Planilha Pagamentos ---> Criação das colunas

In [26]:
import pandas as pd

# Definir data limite
data_limite = pd.to_datetime("2026-04-25")  # ALTERE AQUI

# Ler planilha
df = pd.read_excel("Pagamentos.xlsx")

# Converter coluna de data
df["VencFatal"] = pd.to_datetime(df["VencFatal"], errors="coerce")

# Criar coluna Pagamento
df["Pagamento"] = df["VencFatal"].apply(
    lambda x: "PENDENTE" if x > data_limite else "FEITO"
)

# Criar coluna Valor Restante
df["Valor Restante"] = df.apply(
    lambda row: row["Valor do Lançamento"] if row["VencFatal"] > data_limite else "-",
    axis=1
)

# Salvar resultado
df.to_excel("Pagamentos_atualizado.xlsx", index=False)

print("Planilha atualizada criada com sucesso.")

Planilha atualizada criada com sucesso.
